<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# EduTrack Analytics — DataProjectLab Academy
## Notebook 4 — Dashboard Power BI & Storytelling

> **Prérequis** : Notebooks 1, 2 et 3 complétés. Les fichiers CSV (apprenants nettoyés, scores ML, segmentation) doivent être disponibles.

| | |
|---|---|
| **Niveau** | Avancé |
| **Outils** | Power BI Desktop |
| **Durée estimée** | 6h à 8h |

> 💡 **Ce notebook est un guide de conception reproductible.** En le suivant pas à pas, tu produiras le dashboard exactement tel qu'il apparaît dans le rapport de référence (6 pages : Couverture + Vue executive + Apprenants + Parcours/Instructeurs + Revenus + Alertes ML).

### Objectif business
Transformer les analyses SQL et ML en un dashboard décisionnel **6 pages** permettant à la direction DataProjectLab Academy de piloter : santé globale · profil apprenants · performance pédagogique · revenus · alertes ML préventives.

---
## 1. Sources de données (5 fichiers)

### Fichiers à importer dans Power BI 

| Fichier CSV | Type | Rôle | Volume |
|---|---|---|---|
| `apprenants.csv` | Dimension | Profil apprenants (genre, âge, pays, canal acquisition, premium) | 4 500 lignes |
| `parcours.csv` | Dimension | Catalogue parcours (titre, domaine, niveau, prix, instructeur, note) | 12 lignes |
| `inscriptions_analytics.csv` | Fait | Inscriptions enrichies (statut, progression, CSAT, engagement) | 6 456 lignes |
| `paiements.csv` | Fait | Paiements (montant, méthode, statut) | 8 703 lignes |
| `apprenants_risque_scores.csv` | Fait analytique (ML) | Scores de risque décrochage | 2 095 lignes |

### URLs GitHub raw

```
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/data/apprenants.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/data/parcours.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/data/inscriptions_analytics.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/data/paiements.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/elearning_analytics/data/apprenants_risque_scores.csv



### Import

1. Power BI Desktop → **Obtenir des données → Web**
2. Coller chaque URL dans la fenêtre web
3. Cliquer **Transformer les données** (pour vérifier les types)
4. Vérifier les types en Power Query avant de charger (notamment `premium`, `at_risk_dropout`, `certificat_obtenu`, `alerte_decrochage` en **Nombre entier** 0/1)
5. **Accueil → Fermer & appliquer** 

---
## 2. Désactiver Auto Date/Time (obligatoire)

**Fichier → Options → Chargement des données (Fichier actuel) → DÉCOCHER "Date/heure automatique pour le fichier actuel"**

Sans cette étape, Power BI crée des tables `LocalDateTable_*` parasites pour chaque colonne date qui polluent le modèle et empêchent `PREVIOUSMONTH` / `SAMEPERIODLASTYEAR` de fonctionner.

---
## 3. Modèle de données — Schéma en étoile

### Architecture

```
                          Calendrier (dim temps)
                                 |
                                 v
      apprenants ──> inscriptions_analytics <── parcours
          |                  |                     |
          |                  v                     |
          |             paiements <────────────────┘
          |
          v
      apprenants_risque_scores
```

### 5 relations à créer

| De (N) | Colonne | Vers (1) | Colonne | Cardinalité |
|---|---|---|---|---|
| `inscriptions_analytics` | `apprenant_id` | `apprenants` | `apprenant_id` | N→1 |
| `inscriptions_analytics` | `parcours_id` | `parcours` | `parcours_id` | N→1 |
| `inscriptions_analytics` | `date_inscription` | `Calendrier` | `Date` | N→1 |
| `paiements` | `apprenant_id` | `apprenants` | `apprenant_id` | N→1 |
| `paiements` | `parcours_id` | `parcours` | `parcours_id` | N→1 |
| `apprenants_risque_scores` | `apprenant_id` | `apprenants` | `apprenant_id` | N→1 |

### ⚠️ Points de vigilance

- **Direction : Single** sur toutes les relations
- Si Power BI propose une relation `paiements ↔ inscriptions_analytics` automatiquement (via `apprenant_id` + `parcours_id`), **la supprimer** pour éviter l'ambiguïté de chemin

---
## 4. Table Calendrier

Modélisation → **Nouvelle table** → coller :

```dax
Calendrier =
ADDCOLUMNS(
    CALENDAR(DATE(2022,1,1), DATE(2024,12,31)),
    "Annee",         YEAR([Date]),
    "Mois_Num",      MONTH([Date]),
    "Mois_Nom",      FORMAT([Date], "MMM", "fr-FR"),
    "Mois_Nom_Long", FORMAT([Date], "MMMM", "fr-FR"),
    "Mois_court",    FORMAT([Date], "MMM", "fr-FR"),
    "Annee_Mois",    FORMAT([Date], "YYYY-MM"),
    "Trimestre",     "T" & QUARTER([Date]),
    "Semaine",       WEEKNUM([Date]),
    "Saison",        IF(MONTH([Date]) IN {12,1,2,7,8}, "Haute saison", "Basse saison")
)
```

**Puis marquer comme table de dates** : clic droit `Calendrier` → *Marquer comme table de dates* → colonne `Date`.

### Colonnes calculées additionnelles (pour la heatmap activité de la Page 4)

Sur la table `Calendrier`, ajouter via **Modélisation → Nouvelle colonne** :

```dax
Jour Semaine Nom = 
SWITCH (
    WEEKDAY ( Calendrier[Date], 2 ),
    1, "Lun",  2, "Mar",  3, "Mer",  4, "Jeu",
    5, "Ven",  6, "Sam",  7, "Dim"
)
```

```dax
Jour Semaine Ordre = WEEKDAY ( Calendrier[Date], 2 )
```

Puis trier `Jour Semaine Nom` par `Jour Semaine Ordre` : sélectionner la colonne → onglet **Outils de colonne** → **Trier par colonne** → choisir `Jour Semaine Ordre`.

---
## 5. Table _Mesures (placeholder)

Modélisation → **Nouvelle table** → coller :

```dax
_Mesures = {BLANK()}
```

Puis **masquer la colonne `Value`** (clic droit → Masquer). Toutes les mesures seront rangées dans cette table par dossier d'affichage.

---
## 6. Design system — DataProjectLab Academy

### Palette EduTrack

| Usage | Couleur | Hex |
|---|---|---|
| **Sidebar fond** (page de couverture) | Violet foncé | `#2D2856` |
| **Sidebar fond** (pages dashboard) | Violet profond | `#3A3370` |
| **Accent principal** | Violet | `#5B5BD6` |
| **Item actif sidebar** | Violet vif | `#7C6FD9` |
| Performance positive / Top | Vert | `#1FA67D` (ou `#10B981`) |
| Vigilance / Vue executive (titre) | Violet | `#5B5BD6` |
| Alerte | Rouge | `#E5494D` (ou `#EF4444`) |
| Attention / Année active | Orange | `#F2A93B` (ou `#F59E0B`) |
| CSAT / Performance | Jaune-orange | `#F2A93B` |
| Apprenants (titre Page 2) | Vert turquoise | `#1FA67D` |
| Revenus (titre Page 4) | Vert | `#1FA67D` |
| Parcours (titre Page 3) | Orange | `#F2A93B` |
| Alertes (titre Page 5) | Rouge | `#E5494D` |
| Fond principal pages | Blanc cassé | `#F5F3FA` |
| Fond cartes KPI | Blanc | `#FFFFFF` |
| Texte principal | Gris foncé | `#2C2C2A` |
| Texte secondaire | Gris moyen | `#888780` |

### Typographie

| Usage | Police | Taille |
|---|---|---|
| Titre page (couleur selon page) | Segoe UI **Bold** | 22-24pt |
| Brand `EduTrack Analytics` (couverture) | Segoe UI **Bold** | 72pt |
| Sous-titres | Segoe UI | 13-14pt |
| Valeurs KPI | Segoe UI **Bold** | 38-42pt |
| Labels KPI | Segoe UI | 13pt |
| Sous-labels (italique) | Segoe UI Italic | 11pt |

### Principes

- Fond de page uniforme `#F5F3FA` (gris-violet très clair)
- Cartes KPI avec fond blanc pur, ombre légère, **bordure supérieure 3px colorée** signalant le type de KPI
- Titres de page colorés selon la thématique (violet pour exécutif, vert pour apprenants/revenus, orange pour parcours, rouge pour alertes)

---
## 7. Architecture de navigation — Sidebar DataProjectLab Academy

### Sidebar latérale gauche (présente sur les 5 pages dashboard, pas sur la couverture)

Bande verticale à gauche (largeur ~220px), fond violet `#3A3370`.

#### Contenu en haut

- **Logo DPL** : carré violet 40×40 avec texte "DPL" en blanc bold
- Texte `EduTrack` en blanc **bold** 18pt
- Texte `Analytics` en blanc 13pt regular

#### Menu de navigation (5 items avec icône lettre dans cercle)

| Lettre | Label | Page cible |
|---|---|---|
| **E** | Vue executive | Page 1 |
| **A** | Apprenants | Page 2 |
| **P** | Parcours | Page 3 |
| **R** | Revenus | Page 4 |
| **!** | Alertes ML | Page 5 |

**Item actif** : fond cercle blanc avec lettre violette + texte blanc bold + bordure gauche 3px blanche
**Item inactif** : cercle violet pâle avec lettre blanche + texte blanc opacité 0.7

#### Bouton retour à l'accueil (en bas de sidebar)

Insertion → **Bouton** rond avec icône flèche `←` → Action : **Navigation de page** → cible = **Page 0 (Couverture)**.
Texte sous le bouton : `Retour à l'accueil` en blanc italique 11pt souligné.

#### Bas de sidebar — Signature DPL

- `DataProjectLab` en blanc bold 12pt
- `Academy` en blanc regular 11pt

### Implémentation Power BI

Pour chaque item du menu :
1. Insertion → **Bouton** → texte de l'item
2. Volet Format → Action → **Type : Navigation de page** → cible = page correspondante
3. Pour l'état actif : dupliquer la sidebar sur chaque page et mettre uniquement l'item courant en blanc

---
## 8. Slicers globaux

### 2 slicers en haut à droite des 5 pages dashboard (pas sur la couverture)

#### Slicer 1 — Période (Année)

| Propriété | Valeur |
|---|---|
| Champ | `Calendrier[Annee]` |
| Style | **Boutons horizontaux** |
| Sélection | Single select |
| Valeurs visibles | `2022` · `2023` · `2024` |
| Valeur par défaut | Aucune (= toutes) ou `2024` selon préférence |

**Style** :
- Bouton inactif : fond violet foncé `#5B5BD6`, texte blanc
- Bouton actif : fond orange `#F2A93B`, texte blanc bold
- Forme arrondie (border-radius 4-6px)

#### Slicer 2 — Domaine

| Propriété | Valeur |
|---|---|
| Champ | `parcours[domaine]` |
| Style | **Liste déroulante** |
| Sélection | Multi-select |
| Valeur par défaut | `Tout` |

**Style** : fond blanc, bordure grise fine, label `Domaine :` en gris au-dessus du champ.

### Synchronisation

Clic droit sur chaque slicer → *Synchroniser les segments* → cocher les 5 pages dashboard (visible ET filtre). Décocher la **page de couverture**.

---
## 9. Page 0 — Couverture (page d'accueil)

**Fond plein violet** `#2D2856` avec **2 cercles décoratifs** violet plus clair en haut à droite.

### Éléments de la page

#### Bandeau brand (haut gauche)

Rectangle violet plus clair `#5B5BD6` avec texte `DataProjectLab - Academy` en blanc bold 14pt.

#### Titre brand hero (centre gauche)

Bloc texte massif en 2 lignes :
- `EduTrack` en blanc Segoe UI bold 72pt
- `Analytics` en blanc bold 72pt (ligne suivante)

Sous-titre : `Dashboard Power BI` en violet pâle `#A99AE6` 24pt regular avec une ligne de séparation horizontale fine en dessous.

#### 4 KPI snapshots globaux (bas gauche)

Aligner horizontalement 4 visuels **Carte simple** sans bordure, fond transparent :

| Card | Mesure | Valeur attendue | Couleur |
|---|---|---|---|
| 1 | `[Nb Apprenants Risque Total]` ou `DISTINCTCOUNT(apprenants[apprenant_id])` selon filtres | **3 092** | Blanc |
| 2 | `COUNTROWS(inscriptions_analytics)` | **6 456** | Blanc |
| 3 | `[Taux Completion]` | **33,3%** | Blanc |
| 4 | Format de `[Revenu Genere]` en M FCFA | **749M** *(label "FCFA Rev.")* | Blanc |

Sous chaque valeur, label en violet pâle 14pt regular : `Apprenants` · `Inscriptions` · `Completion` · `FCFA Rev.`

#### Menu navigation horizontal (bas de page)

Bande horizontale violet plus clair `#5B5BD6` avec 5 items texte en blanc :
`Vue executive · Apprenants · Parcours · Revenus · Aletres ML`

Chaque item est un **bouton Navigation de page** vers la page correspondante.

> 💡 La page de couverture n'a **pas de slicer** ni de sidebar — c'est volontaire pour donner un effet "page d'accueil" qui contraste avec le reste du dashboard.

---
## 10. Page 1 — Vue executive

**Titre** : `Vue executive — EduTrack Analytics` en violet `#5B5BD6` Segoe UI Bold 24pt
**Onglet sidebar actif** : Vue executive (E) en blanc
**Slicers visibles** : Période + Domaine

### Ligne 1 : 4 KPI cards

Chaque carte : fond blanc, ombre légère, bordure supérieure 3px colorée, valeur Segoe UI Bold 38pt colorée, label sous la valeur.

| # | Label | Mesure | Valeur attendue | Bordure top | Couleur valeur |
|---|---|---|---|---|---|
| 1 | Total inscriptions | `[Nb Inscriptions]` | **3 846** | 🟣 Violet `#5B5BD6` | Violet `#5B5BD6` |
| 2 | Taux completion *(sous-label : Cible : 40%)* | `[Taux Completion]` | **37,0%** | 🟢 Vert `#1FA67D` | Vert `#1FA67D` |
| 3 | Taux abandon | `[Taux Abandon]` | **19,9%** | 🔴 Rouge `#E5494D` | Rouge `#E5494D` |
| 4 | CSAT moyen *(sous-label : Sur parcours termines)* | `[CSAT Moyen]` | **4,23** | 🟠 Orange `#F2A93B` | Orange `#F2A93B` |

### Ligne 2 : 2 visuels côte à côte

#### Gauche (66%) — Inscriptions mensuelles + taux abandon (combo chart)

**Visuel : Histogramme en colonnes et courbe en regroupement**
- Axe X : `Calendrier[Mois_Nom]`
- Colonnes (Total Inscriptions) : `[Nb Inscriptions]` en violet `#5B5BD6`
- Courbe (Taux Abandon) : `[Taux Abandon]` en rouge `#E5494D` pointillé
- Data labels rouges au-dessus des points (taux abandon mensuel : 20,1% / 19,7% / 17,9% / 20,8% / 20,7% / 20,2% / 22,1% / 20,5% / 18,1% / 19,6% / 21,4% / 17,6%)
- Légende en haut : 🟣 Total Inscriptions · 🔴 Taux Abandon

#### Droite (33%) — Repartition domaines (donut)

**Visuel : Graphique en anneau**
- Légende : `parcours[domaine]`
- Valeurs : `[Nb Inscriptions]`
- **Centre du donut** : afficher `[Taux Completion]` = **37,0%**
- Palette : Data & IA bleu `#3B82F6` · Développement Web violet foncé `#3A3370` · Marketing Digital orange `#F97316` · Management violet `#7C6FD9`

### Ligne 3 : Top 5 parcours par completion vs abandon

**Visuel HTML Content** avec la mesure `Top 5 Parcours Completion vs Abandon HTML`.

Affiche un tableau visuel à 5 lignes avec pour chacun :
- Nom du parcours à gauche
- Barre **Completion** au centre + pourcentage à droite (couleur violette ou verte selon rang)
- Barre **Abandon** à droite + pourcentage rouge

**Top 5 attendu** (trié par taux de complétion DESC) :

| Parcours | Completion | Abandon |
|---|---|---|
| HTML CSS JavaScript | 40,7% 🟣 | 18% |
| SQL & Bases de Donnees | 39,6% 🟣 | 21% |
| Power BI & Dashboards | 39,4% 🟢 | 17% |
| Data Analyse avec Python | 38,1% 🟠 | 20% |
| Machine Learning Applique | 38,0% 🟣 | 21% |

### Installation du visuel HTML Content (si pas déjà fait)

1. Volet Visualisations → `...` → **Obtenir d'autres visuels**
2. Marketplace AppSource → chercher **"HTML Content"** par *Daniel Marsh-Patrick*
3. **Ajouter** → le visuel apparaît dans le panneau
4. Insérer le visuel HTML Content sur la page
5. Glisser la mesure `[Top 5 Parcours Completion vs Abandon HTML]` dans le puits **Values**
6. Format → **Arrière-plan** : désactivé, **Bordure** : désactivée

---
## 11. Page 2 — Profil des apprenants

**Titre** : `Profil des apprenants EduTrack` en vert turquoise `#1FA67D` Segoe UI Bold 24pt
**Onglet sidebar actif** : Apprenants (A)
**Slicers visibles** : Période + Domaine

### Ligne 1 : 3 KPI cards

| # | Label | Mesure | Valeur | Bordure top | Couleur valeur |
|---|---|---|---|---|---|
| 1 | Apprenants | `DISTINCTCOUNT(apprenants[apprenant_id])` (filtré sur année) | **2 144** | 🟢 Vert `#1FA67D` | Vert `#1FA67D` |
| 2 | Apprenants *(sous-label : moyenne d'inscriptions)* | `[Inscriptions par Apprenant]` | **1,4** | 🟢 Vert `#1FA67D` | Vert `#1FA67D` |
| 3 | Apprenants Premium | `[% Apprenants Premium]` | **38%** | 🟣 Violet `#7C6FD9` | Violet `#7C6FD9` |

### Ligne 2 : Apprenants par pays + Distribution des âges

#### Gauche — Apprenants par pays (bar chart horizontal)

**Visuel : Histogramme à barres**
- Axe Y : `Pays Affichage[Pays Affichage]` (table de regroupement Top 5 + Autres — voir section 17)
- Axe X : `[% Apprenants par Pays]`
- Tri : par `Pays Affichage[Ordre]` ASC
- **Couleur par formule** : `[Couleur Pays Affichage]`
- Étiquettes de données : couleur par formule = `[Couleur Pays Affichage]`, position extérieur, format 0%
- Titre du visuel : `Apprenants par pays` en vert `#1FA67D` bold 14pt

**Valeurs attendues** : Côte d'Ivoire 40% 🟢 · Senegal 20% 🟣 · Cameroun 16% 🟠 · Autres 9% ⚪ · Mali 8% 🟪 · Burkina Faso 7% ⚫

#### Droite — Distribution des ages (histogramme à colonnes)

**Visuel : Histogramme à colonnes**
- Axe X : `apprenants[Tranche Age]` trié par `apprenants[Ordre Tranche Age]`
- Axe Y : `[% Distribution Ages]`
- Filtre sur ce visuel : `apprenants[age]` ≥ 18 (exclut les valeurs aberrantes -8, -5, 200)
- **Couleur par formule** : `[Couleur Tranche Age]` (vert pour 23-27 et 28-32, violet pour les autres)
- Étiquettes de données activées avec couleur par formule, position au-dessus, format 0%
- Titre : `Distribution des ages` en vert `#1FA67D` bold 14pt

**Valeurs attendues** : 18-22 = 20% 🟣 · 23-27 = 31% 🟢 · 28-32 = 29% 🟢 · 33-37 = 15% 🟣 · 38-42 = 4% 🟣 · 43+ = 1% 🟣

### Ligne 3 : Canal d'acquisition + Methode de paiement

#### Gauche — Canal d'acquisition (bar chart horizontal)

**Visuel : Histogramme à barres**
- Axe Y : `apprenants[canal_acquisition]`
- Axe X : `[% Apprenants par Canal]`
- Tri : par `% Apprenants par Canal` DESC
- Filtre : `canal_acquisition` n'est pas vide
- **Couleur par formule** : `[Couleur Canal Acquisition]`
- Étiquettes : couleur par formule, format 0%

**Valeurs attendues** : Organique 35% 🟢 · Reseaux sociaux 29% 🟣 · Partenaire 21% 🟠 · Publicite 16% 🔴

#### Droite — Methode de paiement (bar chart horizontal)

**Visuel : Histogramme à barres**
- Axe Y : `paiements[methode]`
- Axe X : `[% Paiements par Methode]`
- Tri : par `% Paiements par Methode` DESC
- **Couleur par formule** : `[Couleur Methode Paiement]`
- Étiquettes : couleur par formule, format 0%

**Valeurs attendues** : Mobile Money 50% 🟢 · Carte bancaire 26% 🟣 · Virement 15% 🟠 · PayPal 10% ⚫

---
## 12. Page 3 — Parcours & Instructeurs (Performance)

**Titre** : `Parcours & Instructeurs — Performance` en orange `#F2A93B` Segoe UI Bold 24pt
**Onglet sidebar actif** : Parcours (P)
**Slicers visibles** : Période + Domaine

### Ligne 1 : Matrice performance instructeurs + Heatmap activité

#### Gauche — Matrice performance instructeurs (scatter)

**Visuel : Nuage de points**

| Puits | Champ / Mesure |
|---|---|
| Valeurs (détails) | `parcours[instructeur]` |
| Axe X | `[CSAT Moyen]` |
| Axe Y | `[Taux Completion]` |
| Taille | `[Nb Inscriptions Instructeur]` |
| Légende | `[Quadrant Instructeur]` |

**Configuration** :
- Filtre : `parcours[instructeur]` n'est pas vide
- Axe X forcé : Min `4,15` · Max `4,35`
- Axe Y forcé : Min `0,34` · Max `0,40`
- Étiquettes de données activées (afficher nom instructeur)
- 4 couleurs assignées manuellement à la légende :
  - Top Performer : violet `#7C6FD9`
  - Coach CSAT : orange `#F2A93B`
  - À accompagner : gris `#9E9E9E`
  - À coacher : vert `#1FA67D`
- Titre : `Matrice performance instructeurs` en orange bold 14pt

**Lecture du quadrant** :
- Haut + droite (Top Performer 🟣) : Ndoye Cheick · Diallo Fatoumata · Traore Eric
- Bas + droite (Coach CSAT 🟠) : Bamba Clarisse · Konaté Aida
- Haut + gauche (À accompagner ⚪) : Camara Luc
- Bas + gauche (À coacher 🟢) : Kouassi Yao · Ouattara Serge

#### Droite — Heatmap activité Sessions Jour × Mois

**Visuel : Matrice native** (avec formatage conditionnel sur la couleur de fond des cellules)

| Puits | Champ |
|---|---|
| Lignes | `Calendrier[Jour Semaine Nom]` (trié par `Jour Semaine Ordre`) |
| Colonnes | `Calendrier[Mois_court]` (trié par `Mois_Num`) |
| Valeurs | `[Nb Inscriptions]` |

**Configuration** :
- Désactiver totaux lignes + colonnes + sous-totaux
- Format → **Éléments de cellule** → Couleur d'arrière-plan → activer → `fx`
  - Style : Dégradé
  - Champ : `[Nb Inscriptions]`
  - Couleur min : lavande pâle `#F5F3FF`
  - Couleur max : violet foncé `#4C3EB8`
- Format → **Éléments de cellule** → Couleur de la police → activer → `fx`
  - Style : Dégradé
  - Couleur min : gris foncé `#666666` · Couleur max : blanc `#FFFFFF`
- Titre : `Heatmap activite — Sessions heure x jour` en orange bold 14pt

> ⚠️ **Note importante** : la heatmap utilise **Jour × Mois** (les données ne contiennent pas d'information d'heure). Le titre `Sessions heure x jour` est préservé tel qu'il apparaît dans le rapport, mais il s'agit en réalité d'une heatmap **Jour × Mois**.

### Ligne 2 : Taux abandon par domaine & parcours (4 cards)

Pour chacun des 4 domaines (Data & IA, Dev Web, Marketing Dig., Management), créer **un visuel Carte multi-lignes** avec :
- 1ère ligne : `parcours[domaine]` (titre du domaine en bold 14pt)
- 2ème ligne : `[Taux Abandon]` (en bold 16pt couleur selon domaine) + label `abandon moy.`
- 3ème ligne : `[Pire Parcours Label]` (en italique gris 12pt)

**Filtre sur chaque card** : `parcours[domaine]` = un seul domaine

**Bordure supérieure 3px colorée** :

| Domaine | Bordure top | Couleur valeur abandon | Pire parcours attendu |
|---|---|---|---|
| Data & IA | 🔴 Rouge `#E5494D` | Rouge | SQL & BDD 21,4% |
| Dev Web | 🟠 Orange `#F2A93B` | Orange | Python Django 20,1% |
| Marketing Dig. | 🟠 Orange `#F2A93B` | Orange | Marketing Dig. 21,9% |
| Management | 🔴 Rouge `#E5494D` | Rouge | Mgmt Projet 19,8% |

**Titre commun au-dessus** : `Taux abandon par domaine & parcours` en orange `#F2A93B` Segoe UI Bold 16pt

---
## 13. Page 4 — Revenus & Paiements

**Titre** : `Revenus & Paiements EduTrack` en vert `#1FA67D` Segoe UI Bold 24pt
**Onglet sidebar actif** : Revenus (R)
**Slicers visibles** : Période + Domaine

### Ligne 1 : 3 KPI cards

| # | Label | Mesure | Valeur | Bordure top | Couleur valeur |
|---|---|---|---|---|---|
| 1 | FCFA Revenue total | `[Revenu Genere]` formaté en M FCFA | **749M** | 🟢 Vert `#1FA67D` | Vert `#1FA67D` |
| 2 | Equivalent EURO | `[Revenu Genere]` ÷ 655 ÷ 1000000 | **€ 1,15M** | 🟣 Violet `#5B5BD6` | Violet `#5B5BD6` |
| 3 | FCFA panier moyen | `DIVIDE([Revenu Genere], COUNTROWS(paiements))` | **89 554** | 🟠 Orange `#F2A93B` | Vert `#1FA67D` |

> 💡 **Mesure suggérée pour KPI 2** :
> ```dax
> Revenu EUR = DIVIDE([Revenu Genere], 655.957) -- taux fixe FCFA→EUR
> ```

### Ligne 2 : Revenus mensuels par domaine + Méthodes de paiement

#### Gauche (66%) — Revenus mensuels par domaine (FCFA) (stacked bar)

**Visuel : Histogramme à colonnes empilées**
- Axe X : `Calendrier[Mois_court]` (trié par `Mois_Num`)
- Axe Y (valeurs) : `[Revenu Genere]`
- Légende : `parcours[domaine]`
- Palette par domaine :
  - Data & IA : violet foncé `#3A3370`
  - Développement Web : vert `#1FA67D`
  - Management : orange `#F2A93B`
  - Marketing Digital : violet pâle `#A99AE6`
- Légende en bas du visuel
- Titre : `Revenus mensuels par domaine (FCFA)` en vert bold 14pt

#### Droite (33%) — Méthodes de paiement (bar chart horizontal)

**Visuel : Histogramme à barres**
- Axe Y : `paiements[methode]`
- Axe X : `[Revenu Genere]`
- Tri : par `Revenu Genere` DESC
- **Couleur par formule** : `[Couleur Methode Paiement]`
- Étiquettes : format `0,0,, "M FCFA"` (les 2 virgules divisent par 1 million)
- Titre : `Methodes de paiement` en vert bold 14pt

**Valeurs attendues** : Mobile Money 380M 🟢 · Carte bancaire 204M 🟣 · Virement 115M 🟠 · PayPal 78M ⚫

### Ligne 3 : Top 5 parcours par revenu généré (5 cards)

Pour chaque rang (1 à 5), créer **3 visuels Carte simples empilés** dans un cadre avec bordure supérieure colorée.

#### Structure de chaque card (3 cartes simples empilées)

| Position | Champ / Mesure | Format |
|---|---|---|
| Card titre (haut) | `[TopN Nom]` | Segoe UI Bold 13pt · gris foncé |
| Card valeur (milieu) | `[TopN Revenu]` | Segoe UI Bold 18pt · couleur du rang |
| Card sous-texte (bas) | `[TopN Inscrits]` | Segoe UI Italic 11pt · gris |

Pour chaque card, mettre **étiquette de catégorie : désactivée** et **arrière-plan : transparent**.

Insérer un **rectangle blanc** derrière les 3 cards pour servir de cadre + bordure top colorée 3px.

#### Code couleur des bordures et valeurs

| Rang | Parcours | Bordure top + Couleur valeur |
|---|---|---|
| 1 | React & Node.js Fullstack | 🟢 Vert `#1FA67D` |
| 2 | Machine Learning Applique | 🟣 Violet `#7C6FD9` |
| 3 | Python Web avec Django | 🟠 Orange `#F2A93B` |
| 4 | Data Analyse avec Python | 🟣 Violet `#7C6FD9` |
| 5 | Power BI & Dashboards | 🟢 Vert `#1FA67D` |

**Valeurs attendues** :
- React & Node.js Fullstack — **120M FCFA** — *297 inscrits*
- Machine Learning Applique — **109M FCFA** — *355 inscrits*
- Python Web avec Django — **94M FCFA** — *334 inscrits*
- Data Analyse avec Python — **72M FCFA** — *318 inscrits*
- Power BI & Dashboards — **64M FCFA** — *312 inscrits*

**Titre commun au-dessus** : `Top 5 parcours par revenu genere (FCFA)` en vert turquoise `#1FA67D` Segoe UI Bold 16pt

---
## 14. Page 5 — Alertes ML — Détection du décrochage

**Titre** : `Alertes ML — Detection du decrochage` en rouge `#E5494D` Segoe UI Bold 24pt
**Onglet sidebar actif** : Alertes ML (!) en rouge
**Slicers visibles** : Domaine uniquement (pas de slicer Période ici)

### Ligne 1 : Bandeau d'alerte (carte texte + badge)

Bandeau rouge clair `#FCE5E5` avec bordure rouge sur toute la largeur, contenant :

#### Gauche (75%) — Phrase d'alerte
**Visuel : Carte simple**
- Champ : `[Phrase Alerte Decrochage]`
- Format : Segoe UI Bold 16-18pt · couleur rouge `#E5494D`
- Étiquette de catégorie : désactivée

Texte affiché : *"**1 613 apprenants en alerte de decrochage sur 2 095 en cours**"*

#### Droite (25%) — Badge taux d'alerte
**Visuel : Carte simple**
- Champ : `[Taux Alerte Decrochage Label]`
- Format : Segoe UI Bold 14pt · couleur **blanche**
- Couleur d'arrière-plan : rouge `#E5494D`
- Coins arrondis · centré

Texte affiché : **77,0% taux d'alerte**

### Ligne 2 : 3 actions de prévention (cards)

Pour chaque carte, structure : icône cercle coloré à gauche + titre `Score X-Y` + label action + sous-texte avec nombre d'apprenants approximatif.

#### Card 1 — Appel téléphonique (rouge)
- **Icône** : cercle rouge `#E5494D` avec `!` blanc
- **Titre** : `Score > 0.80` en rouge bold
- **Sous-titre** : `Appel telephonique` en violet
- **Sous-texte** : `[Label Apprenants Score > 0.80]` → **~150 apprenants** en italique rouge
- Bordure top 3px rouge
- Fond rose pâle `#FCE5E5`

#### Card 2 — Email personnalisé (orange)
- **Icône** : cercle orange `#F2A93B` avec `@` blanc
- **Titre** : `Score 0.60 – 0.80` en orange bold
- **Sous-titre** : `Email personnalise` en violet
- **Sous-texte** : `[Label Apprenants Score 0.60 - 0.80]` → **~150 apprenants** en italique orange
- Bordure top 3px orange
- Fond orange pâle `#FEF1DC`

#### Card 3 — Notification push (violet)
- **Icône** : cercle violet foncé `#3A3370` avec `n` blanc
- **Titre** : `Score 0.30 – 0.60` en violet bold
- **Sous-titre** : `Notification push` en violet
- **Sous-texte** : `[Label Apprenants Score 0.30 - 0.60]` → **~1 300 apprenants** en italique violet
- Bordure top 3px violette
- Fond lavande `#EEEEFA`

> 💡 **Note** : la valeur réelle sur Card 3 est ~1 300 apprenants (1 299 exactement). Sur les Cards 1 et 2, les valeurs réelles sont 163 et 151 — toutes deux arrondies à 150 par la mesure d'arrondi à la 50aine.

### Ligne 3 : Top apprenants à risque — Tableau d'alerte

**Titre** : `Top apprenants a risque — Tableau d'alerte` en rouge `#E5494D` Segoe UI Bold 16pt

**Visuel : Table** native

| Colonne | Champ / Mesure | Formatage |
|---|---|---|
| Apprenant | `apprenants_risque_scores[Apprenant Nom Complet]` | Bold 12pt |
| Parcours | `apprenants_risque_scores[Parcours Court]` | Regular |
| Progression | `[Progression Label]` | Format texte (ex. `15 %`) |
| Engagement | `apprenants_risque_scores[engagement_score]` | 1 décimale |
| Inactivite | `[Inactivite Label]` | Format texte (ex. `68j`) |
| Score risque | `apprenants_risque_scores[score_risque]` | 2 décimales · couleur par formule = `[Couleur Score Risque]` |
| Action | `[Action Recommandee]` | Badge coloré (couleur fond = `[Couleur Action]`, police blanche) |

**Configuration** :
- Filtre : `score_risque` ≥ 0.30 (uniquement les apprenants nécessitant une action)
- Tri : `score_risque` DESC
- En-tête : fond bleu marine `#1F2547`, police blanche bold
- Lignes alternées : `#FFFFFF` / `#FAFAFA`

**Formatage conditionnel** :
- Sur la colonne **Score risque** : Format → Éléments de cellule → Couleur de la police → `fx` → Valeur de champ → `[Couleur Score Risque]`
- Sur la colonne **Action** : Couleur d'arrière-plan → `fx` → `[Couleur Action]` + Couleur de la police → forcer `#FFFFFF`

---
## 15. Mesures DAX — KPIs de base (5 mesures)

Toutes les mesures sont créées dans la table `_Mesures`, organisées par dossier d'affichage.

### 📂 Dossier `Pédagogie`

```dax
Nb Inscriptions = COUNTROWS ( inscriptions_analytics )

Taux Completion = 
VAR _termine = CALCULATE ( COUNTROWS ( inscriptions_analytics ), inscriptions_analytics[statut] = "Termine" )
VAR _total = COUNTROWS ( inscriptions_analytics )
RETURN DIVIDE ( _termine, _total )

Taux Abandon = 
VAR _abandon = CALCULATE ( COUNTROWS ( inscriptions_analytics ), inscriptions_analytics[statut] = "Abandonne" )
VAR _total = COUNTROWS ( inscriptions_analytics )
RETURN DIVIDE ( _abandon, _total )

CSAT Moyen = AVERAGE ( inscriptions_analytics[csat] )

Nb Inscriptions Instructeur = COUNTROWS ( inscriptions_analytics )
```

**Format** : `Nb Inscriptions` et `Nb Inscriptions Instructeur` en `#,0` · `Taux Completion`, `Taux Abandon` en `0.0%` · `CSAT Moyen` en `0.00`

---
## 16. Mesures DAX — Démographie & Acquisition (8 mesures)

### 📂 Dossier `Démographie`

```dax
% Apprenants Premium = 
VAR _premium = CALCULATE ( COUNTROWS ( apprenants ), apprenants[premium] = 1 )
VAR _total = COUNTROWS ( apprenants )
RETURN DIVIDE ( _premium, _total )

Inscriptions par Apprenant = 
DIVIDE (
    COUNTROWS ( inscriptions_analytics ),
    DISTINCTCOUNT ( inscriptions_analytics[apprenant_id] )
)

% Distribution Ages = 
VAR _curr =
    CALCULATE ( COUNTROWS ( apprenants ), apprenants[age] >= 18 )
VAR _total =
    CALCULATE (
        COUNTROWS ( apprenants ),
        ALL ( apprenants[Tranche Age] ),
        ALL ( apprenants[Ordre Tranche Age] ),
        apprenants[age] >= 18
    )
RETURN DIVIDE ( _curr, _total )

Couleur Tranche Age = 
VAR _ordreCurr = SELECTEDVALUE ( apprenants[Ordre Tranche Age] )
RETURN IF ( _ordreCurr IN { 2, 3 }, "#1FA67D", "#7C6FD9" )

% Apprenants par Pays = 
VAR _pays_courant = SELECTEDVALUE ( 'Pays Affichage'[Pays Affichage] )
VAR _total = CALCULATE ( COUNTROWS ( apprenants ), ALL ( apprenants ) )
VAR _top5 = { "Cote d'Ivoire", "Senegal", "Cameroun", "Mali", "Burkina Faso" }
VAR _nb =
    SWITCH (
        TRUE (),
        _pays_courant = "Autres",
            CALCULATE ( COUNTROWS ( apprenants ), ALL ( apprenants ), NOT ( apprenants[pays] IN _top5 ) ),
        CALCULATE ( COUNTROWS ( apprenants ), ALL ( apprenants ), apprenants[pays] = _pays_courant )
    )
RETURN DIVIDE ( _nb, _total )

Couleur Pays Affichage = SELECTEDVALUE ( 'Pays Affichage'[Couleur] )
```

### 📂 Dossier `Acquisition`

```dax
% Apprenants par Canal = 
VAR _canal_courant = SELECTEDVALUE ( apprenants[canal_acquisition] )
VAR _curr =
    CALCULATE (
        COUNTROWS ( apprenants ),
        ALL ( apprenants ),
        apprenants[canal_acquisition] = _canal_courant
    )
VAR _total =
    CALCULATE (
        COUNTROWS ( apprenants ),
        ALL ( apprenants ),
        apprenants[canal_acquisition] <> BLANK ()
    )
RETURN DIVIDE ( _curr, _total )

Couleur Canal Acquisition = 
SWITCH (
    SELECTEDVALUE ( apprenants[canal_acquisition] ),
    "Organique",       "#1FA67D",
    "Reseaux sociaux", "#5B5BD6",
    "Partenaire",      "#F2A93B",
    "Publicite",       "#E5494D",
    "#BDBDBD"
)
```

**Format** : `% Apprenants Premium`, `% Distribution Ages`, `% Apprenants par Pays`, `% Apprenants par Canal` en `0%` · `Inscriptions par Apprenant` en `0.0`

---
## 17. Table de regroupement `Pays Affichage`

Pour afficher Top 5 pays + ligne agrégée "Autres" sur le bar chart Apprenants par pays, on utilise une table calculée autonome (pas de relation avec apprenants).

Modélisation → **Nouvelle table** → coller :

```dax
Pays Affichage =
DATATABLE (
    "Pays Affichage", STRING,
    "Ordre", INTEGER,
    "Couleur", STRING,
    {
        { "Cote d'Ivoire",  1, "#1FA67D" },
        { "Senegal",        2, "#5B5BD6" },
        { "Cameroun",       3, "#F2A93B" },
        { "Mali",           4, "#B9B5F0" },
        { "Burkina Faso",   5, "#7A7A7A" },
        { "Autres",         6, "#BDBDBD" }
    }
)
```

Puis trier la colonne `Pays Affichage` par `Ordre` : sélectionner la colonne → onglet **Outils de colonne** → **Trier par colonne** → choisir `Ordre`.

> 💡 La table `Pays Affichage` n'a pas besoin de relation avec `apprenants` — la mesure `[% Apprenants par Pays]` fait le filtrage elle-même via `SELECTEDVALUE`.

---
## 18. Mesures DAX — Paiement & Revenus (4 mesures)

### 📂 Dossier `Paiement`

```dax
% Paiements par Methode = 
VAR _methode_courante = SELECTEDVALUE ( paiements[methode] )
VAR _curr =
    CALCULATE (
        COUNTROWS ( paiements ),
        ALL ( paiements ),
        paiements[methode] = _methode_courante
    )
VAR _total = CALCULATE ( COUNTROWS ( paiements ), ALL ( paiements ) )
RETURN DIVIDE ( _curr, _total )

Couleur Methode Paiement = 
SWITCH (
    SELECTEDVALUE ( paiements[methode] ),
    "Mobile Money",   "#1FA67D",
    "Carte bancaire", "#5B5BD6",
    "Virement",       "#F2A93B",
    "PayPal",         "#7A7A7A",
    "#BDBDBD"
)
```

### 📂 Dossier `Business`

```dax
Revenu Genere = SUM ( paiements[montant_fcfa] )

Top 5 Parcours Rang = 
RANKX (
    ALL ( parcours[titre] ),
    [Revenu Genere],
    ,
    DESC,
    DENSE
)
```

**Format** : `% Paiements par Methode` en `0%` · `Revenu Genere` en `#,0" FCFA"` · `Top 5 Parcours Rang` en `0`

---
## 19. Mesures DAX — Top 5 Parcours (15 mesures pour la Page 4)

### 📂 Dossier `Top 5 Parcours`

Chaque rang (1 à 5) a 3 mesures : **Nom**, **Revenu**, **Inscrits**. La logique est identique, seule la valeur du rang change.

#### Pattern réutilisable (à dupliquer pour les rangs 1 à 5)

```dax
TopN Nom = 
VAR _ranked =
    ADDCOLUMNS (
        VALUES ( parcours[titre] ),
        "@rev", [Revenu Genere],
        "@rang",
            RANKX ( ALL ( parcours[titre] ), [Revenu Genere], , DESC, DENSE )
    )
VAR _row = FILTER ( _ranked, [@rang] = N )   -- Remplacer N par 1, 2, 3, 4 ou 5
RETURN MAXX ( _row, parcours[titre] )

TopN Revenu = 
VAR _ranked =
    ADDCOLUMNS (
        VALUES ( parcours[titre] ),
        "@rev", [Revenu Genere],
        "@rang",
            RANKX ( ALL ( parcours[titre] ), [Revenu Genere], , DESC, DENSE )
    )
VAR _row = FILTER ( _ranked, [@rang] = N )
VAR _r = MAXX ( _row, [@rev] )
RETURN
    SWITCH (
        TRUE (),
        _r >= 1000000, FORMAT ( _r / 1000000, "0" ) & "M FCFA",
        FORMAT ( _r / 1000, "0" ) & "k FCFA"
    )

TopN Inscrits = 
VAR _ranked =
    ADDCOLUMNS (
        VALUES ( parcours[titre] ),
        "@rev", [Revenu Genere],
        "@rang",
            RANKX ( ALL ( parcours[titre] ), [Revenu Genere], , DESC, DENSE )
    )
VAR _row = FILTER ( _ranked, [@rang] = N )
VAR _nom = MAXX ( _row, parcours[titre] )
VAR _nb =
    CALCULATE (
        COUNTROWS ( inscriptions_analytics ),
        ALL ( parcours ),
        parcours[titre] = _nom
    )
RETURN FORMAT ( _nb, "#,0" ) & " inscrits"
```

Créer **15 mesures au total** : `Top1 Nom`, `Top1 Revenu`, `Top1 Inscrits`, ..., `Top5 Nom`, `Top5 Revenu`, `Top5 Inscrits`. Pour chacune, remplacer `N` par le rang voulu dans la ligne `[@rang] = N`.

### Valeurs attendues

| Mesure | Valeur |
|---|---|
| `Top1 Nom` / `Top1 Revenu` / `Top1 Inscrits` | React & Node.js Fullstack · 120M FCFA · 297 inscrits |
| `Top2 Nom` / `Top2 Revenu` / `Top2 Inscrits` | Machine Learning Applique · 109M FCFA · 355 inscrits |
| `Top3 Nom` / `Top3 Revenu` / `Top3 Inscrits` | Python Web avec Django · 94M FCFA · 334 inscrits |
| `Top4 Nom` / `Top4 Revenu` / `Top4 Inscrits` | Data Analyse avec Python · 72M FCFA · 318 inscrits |
| `Top5 Nom` / `Top5 Revenu` / `Top5 Inscrits` | Power BI & Dashboards · 64M FCFA · 312 inscrits |

---
## 20. Mesures DAX — Performance Instructeurs (Page 3) — 2 mesures

### 📂 Dossier `Pédagogie`

```dax
Quadrant Instructeur = 
VAR _csat = [CSAT Moyen]
VAR _comp = [Taux Completion]
VAR _csat_seuil = 4.23
VAR _comp_seuil = 0.333
RETURN
    SWITCH (
        TRUE (),
        _comp > _comp_seuil && _csat > _csat_seuil,  "Top Performer",
        _comp > _comp_seuil && _csat <= _csat_seuil, "A accompagner",
        _comp <= _comp_seuil && _csat > _csat_seuil, "Coach CSAT",
        "A coacher"
    )
```

### 📂 Dossier `Pédagogie`

```dax
Pire Parcours du Domaine = 
VAR _tbl =
    ADDCOLUMNS (
        VALUES ( parcours[titre] ),
        "@tx", [Taux Abandon]
    )
VAR _maxTx = MAXX ( _tbl, [@tx] )
VAR _topRow = TOPN ( 1, FILTER ( _tbl, [@tx] = _maxTx ), [@tx], DESC )
RETURN MAXX ( _topRow, parcours[titre] )

Pire Parcours Taux = 
VAR _tbl =
    ADDCOLUMNS (
        VALUES ( parcours[titre] ),
        "@tx", [Taux Abandon]
    )
RETURN MAXX ( _tbl, [@tx] )

Pire Parcours Label = 
VAR _nom = [Pire Parcours du Domaine]
VAR _tx = [Pire Parcours Taux]
VAR _nom_court =
    SWITCH (
        TRUE (),
        _nom = "Machine Learning Applique",     "Machine Learning",
        _nom = "React & Node.js Fullstack",     "React & Node.js",
        _nom = "SEO & Growth Hacking",          "SEO & Growth",
        _nom = "Management de Projet",          "Mgmt Projet",
        _nom = "HTML CSS JavaScript",           "HTML CSS JS",
        _nom = "Python Web avec Django",        "Python Django",
        _nom = "Data Analyse avec Python",      "Data Python",
        _nom = "Power BI & Dashboards",         "Power BI",
        _nom = "SQL & Bases de Donnees",        "SQL & BDD",
        _nom = "Leadership & Soft Skills",      "Leadership",
        _nom = "Community Management",          "Community Mgmt",
        _nom = "Marketing Digital",             "Marketing Dig.",
        _nom
    )
RETURN _nom_court & " " & FORMAT ( _tx, "0.0%" )
```

---
## 21. Mesures DAX — Alertes ML (Page 5) — 13 mesures

### 📂 Dossier `Alerte Décrochage`

#### Bandeau d'alerte

```dax
Nb Apprenants Alerte = 
CALCULATE (
    COUNTROWS ( apprenants_risque_scores ),
    apprenants_risque_scores[alerte_decrochage] = 1
)

Nb Apprenants Risque Total = COUNTROWS ( apprenants_risque_scores )

Phrase Alerte Decrochage = 
VAR _alerte = [Nb Apprenants Alerte]
VAR _total = [Nb Apprenants Risque Total]
RETURN
    FORMAT ( _alerte, "#,0" ) & " apprenants en alerte de decrochage sur "
        & FORMAT ( _total, "#,0" ) & " en cours"

Taux Alerte Decrochage Label = 
VAR _taux = DIVIDE ( [Nb Apprenants Alerte], [Nb Apprenants Risque Total] )
RETURN FORMAT ( _taux, "0.0%" ) & " taux d'alerte"
```

#### 3 KPIs d'actions de prévention

```dax
Nb Apprenants Score > 0.80 = 
CALCULATE (
    COUNTROWS ( apprenants_risque_scores ),
    apprenants_risque_scores[score_risque] > 0.80
)

Nb Apprenants Score 0.60 - 0.80 = 
CALCULATE (
    COUNTROWS ( apprenants_risque_scores ),
    apprenants_risque_scores[score_risque] >= 0.60,
    apprenants_risque_scores[score_risque] <= 0.80
)

Nb Apprenants Score 0.30 - 0.60 = 
CALCULATE (
    COUNTROWS ( apprenants_risque_scores ),
    apprenants_risque_scores[score_risque] >= 0.30,
    apprenants_risque_scores[score_risque] < 0.60
)

Label Apprenants Score > 0.80 = 
VAR _nb = [Nb Apprenants Score > 0.80]
VAR _arrondi = ROUND ( _nb / 50, 0 ) * 50
RETURN "~" & FORMAT ( _arrondi, "#,0" ) & " apprenants"

Label Apprenants Score 0.60 - 0.80 = 
VAR _nb = [Nb Apprenants Score 0.60 - 0.80]
VAR _arrondi = ROUND ( _nb / 50, 0 ) * 50
RETURN "~" & FORMAT ( _arrondi, "#,0" ) & " apprenants"

Label Apprenants Score 0.30 - 0.60 = 
VAR _nb = [Nb Apprenants Score 0.30 - 0.60]
VAR _arrondi = ROUND ( _nb / 50, 0 ) * 50
RETURN "~" & FORMAT ( _arrondi, "#,0" ) & " apprenants"
```

#### Tableau Top apprenants à risque

```dax
Action Recommandee = 
VAR _score = SELECTEDVALUE ( apprenants_risque_scores[score_risque] )
RETURN
    SWITCH (
        TRUE (),
        _score > 0.80,                        "Appel urgent",
        _score >= 0.60 && _score <= 0.80,    "Email perso",
        _score >= 0.30 && _score < 0.60,     "Push",
        "RAS"
    )

Couleur Action = 
SWITCH (
    [Action Recommandee],
    "Appel urgent", "#E5494D",
    "Email perso",  "#F2A93B",
    "Push",         "#7C6FD9",
    "#9CA3AF"
)

Couleur Score Risque = 
VAR _score = SELECTEDVALUE ( apprenants_risque_scores[score_risque] )
RETURN
    SWITCH (
        TRUE (),
        _score > 0.80,  "#E5494D",
        _score >= 0.60, "#F2A93B",
        _score >= 0.30, "#7C6FD9",
        "#888888"
    )

Inactivite Label = 
FORMAT ( SELECTEDVALUE ( apprenants_risque_scores[nb_jours_inactif] ), "#,0" ) & "j"

Progression Label = 
FORMAT ( SELECTEDVALUE ( apprenants_risque_scores[progression_pct] ), "0" ) & "%"
```

---
## 22. Colonnes calculées (4 colonnes)

### Sur la table `apprenants`

Ces colonnes alimentent l'histogramme **Distribution des âges** de la Page 2.

```dax
Tranche Age = 
VAR _age = apprenants[age]
RETURN
    SWITCH (
        TRUE (),
        _age >= 18 && _age <= 22, "18-22",
        _age >= 23 && _age <= 27, "23-27",
        _age >= 28 && _age <= 32, "28-32",
        _age >= 33 && _age <= 37, "33-37",
        _age >= 38 && _age <= 42, "38-42",
        _age >= 43, "43+",
        "Hors plage"
    )

Ordre Tranche Age = 
VAR _age = apprenants[age]
RETURN
    SWITCH (
        TRUE (),
        _age >= 18 && _age <= 22, 1,
        _age >= 23 && _age <= 27, 2,
        _age >= 28 && _age <= 32, 3,
        _age >= 33 && _age <= 37, 4,
        _age >= 38 && _age <= 42, 5,
        _age >= 43, 6,
        99
    )
```

**Tri** : sélectionner `Tranche Age` → onglet **Outils de colonne** → **Trier par colonne** → choisir `Ordre Tranche Age`.

### Sur la table `apprenants_risque_scores`

Ces colonnes alimentent le tableau **Top apprenants à risque** de la Page 5.

```dax
Apprenant Nom Complet = 
apprenants_risque_scores[prenom] & " " & apprenants_risque_scores[nom]

Parcours Court = 
VAR _t = apprenants_risque_scores[titre]
RETURN
    SWITCH (
        TRUE (),
        _t = "React & Node.js Fullstack",     "React & Node.js",
        _t = "Machine Learning Applique",     "Machine Learning",
        _t = "Python Web avec Django",        "Python Web",
        _t = "Data Analyse avec Python",      "Data Analyse",
        _t = "Power BI & Dashboards",         "Power BI",
        _t = "SQL & Bases de Donnees",        "SQL & BDD",
        _t = "HTML CSS JavaScript",           "HTML CSS JS",
        _t = "Management de Projet",          "Management Projet",
        _t = "Leadership & Soft Skills",      "Leadership",
        _t = "Marketing Digital",             "Marketing Digital",
        _t = "Community Management",          "Community Mgmt",
        _t = "SEO & Growth Hacking",          "SEO & Growth",
        _t
    )
```

---
## 23. Mesures DAX — Dossier Variations vs N-1 (8 mesures)

```dax
Variation Inscriptions % = 
VAR _curr = [Nb Inscriptions]
VAR _prev = CALCULATE([Nb Inscriptions], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _pct = DIVIDE(_curr - _prev, _prev)
VAR _f = FORMAT(_pct, "+0%;-0%")
RETURN
    SWITCH(
        TRUE(),
        _pct > 0, UNICHAR(9650) & " " & _f & " vs 2023",
        _pct < 0, UNICHAR(9660) & " " & _f & " vs 2023",
        UNICHAR(9650) & " 0% vs 2023"
    )

Variation Completion vs Cible = 
VAR _curr = [Taux Completion]
VAR _cible = 0.40
VAR _delta = (_curr - _cible) * 100
VAR _f = FORMAT(_delta, "+0;-0") & "pp vs cible 40%"
RETURN
    SWITCH(
        TRUE(),
        _delta > 0, UNICHAR(9650) & " " & _f,
        _delta < 0, UNICHAR(9660) & " " & _f,
        UNICHAR(9650) & " 0pp vs cible 40%"
    )

Variation Abandon pp = 
VAR _curr = [Taux Abandon]
VAR _prev = CALCULATE([Taux Abandon], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _delta = (_curr - _prev) * 100
VAR _f = FORMAT(_delta, "+0.0;-0.0") & "pp vs 2023"
RETURN
    SWITCH(
        TRUE(),
        _delta > 0, UNICHAR(9650) & " " & _f,
        _delta < 0, UNICHAR(9660) & " " & _f,
        UNICHAR(9650) & " 0pp vs 2023"
    )

Variation CSAT = 
VAR _curr = [CSAT Moyen]
VAR _prev = CALCULATE([CSAT Moyen], SAMEPERIODLASTYEAR(Calendrier[Date]))
VAR _delta = _curr - _prev
VAR _f = FORMAT(_delta, "+0.0;-0.0") & " vs 2023"
RETURN
    SWITCH(
        TRUE(),
        _delta > 0, UNICHAR(9650) & " " & _f,
        _delta < 0, UNICHAR(9660) & " " & _f,
        UNICHAR(9650) & " 0 vs 2023"
    )
```

---
## 24. Mesure HTML — Top 5 Parcours Completion vs Abandon (Page 1)

Cette mesure retourne du HTML à utiliser avec le visuel **HTML Content** pour afficher le tableau visuel comparatif Top 5 parcours par taux de complétion.

### 📂 Dossier `HTML Content`

```dax
Top 5 Parcours Completion vs Abandon HTML = 
VAR _tbl =
    ADDCOLUMNS (
        VALUES ( parcours[titre] ),
        "total", CALCULATE ( COUNTROWS ( inscriptions_analytics ) ),
        "tx_comp",
            DIVIDE (
                CALCULATE ( COUNTROWS ( inscriptions_analytics ),
                            inscriptions_analytics[statut] = "Termine" ),
                CALCULATE ( COUNTROWS ( inscriptions_analytics ) )
            ),
        "tx_aband",
            DIVIDE (
                CALCULATE ( COUNTROWS ( inscriptions_analytics ),
                            inscriptions_analytics[statut] = "Abandonne" ),
                CALCULATE ( COUNTROWS ( inscriptions_analytics ) )
            )
    )
VAR _top5 = TOPN ( 5, FILTER ( _tbl, [total] > 0 ), [tx_comp], DESC )
VAR _ranked = ADDCOLUMNS ( _top5, "rang", RANKX ( _top5, [tx_comp], , DESC ) )
VAR _max_comp = MAXX ( _top5, [tx_comp] )
VAR _max_aband = MAXX ( _top5, [tx_aband] )
VAR _barMaxPx = 220
VAR _nameW = 180
VAR _val_col_w = 55
VAR _header =
    "<div style='display:flex;align-items:center;margin-bottom:12px;padding-bottom:8px;border-bottom:1px solid #E5E7EB;'>" &
        "<div style='width:" & _nameW & "px;'></div>" &
        "<div style='width:" & _barMaxPx & "px;text-align:center;color:#9CA3AF;font-size:12px;font-weight:500;margin:0 14px;'>Completion</div>" &
        "<div style='width:" & _val_col_w & "px;'></div>" &
        "<div style='width:" & _barMaxPx & "px;text-align:center;color:#9CA3AF;font-size:12px;font-weight:500;margin:0 14px;'>Abandon</div>" &
        "<div style='width:" & _val_col_w & "px;'></div>" &
    "</div>"
VAR _rows =
    CONCATENATEX (
        _ranked,
        VAR _t = [titre]
        VAR _c = [tx_comp]
        VAR _a = [tx_aband]
        VAR _r = [rang]
        VAR _pxC = ROUND ( DIVIDE ( _c, _max_comp ) * _barMaxPx, 0 )
        VAR _pxA = ROUND ( DIVIDE ( _a, _max_aband ) * _barMaxPx, 0 )
        VAR _colC =
            SWITCH (
                TRUE (),
                _r = 3, "#10B981",
                _r = 4, "#F59E0B",
                "#4F46E5"
            )
        VAR _fmtC = FORMAT ( _c, "0.0%" )
        VAR _fmtA = FORMAT ( _a, "0%" )
        RETURN
            "<div style='display:flex;align-items:center;margin-bottom:14px;'>" &
                "<div style='width:" & _nameW & "px;color:#111827;font-size:13px;font-weight:700;padding-right:12px;'>" & _t & "</div>" &
                "<div style='width:" & _barMaxPx & "px;background:#EEF2FF;height:18px;border-radius:2px;margin:0 14px;'>" &
                    "<div style='background:" & _colC & ";height:100%;width:" & _pxC & "px;border-radius:2px;'></div>" &
                "</div>" &
                "<div style='width:" & _val_col_w & "px;color:" & _colC & ";font-size:14px;font-weight:700;text-align:left;'>" & _fmtC & "</div>" &
                "<div style='width:" & _barMaxPx & "px;background:#FEE2E2;height:18px;border-radius:2px;margin:0 14px;'>" &
                    "<div style='background:#F87171;height:100%;width:" & _pxA & "px;border-radius:2px;'></div>" &
                "</div>" &
                "<div style='width:" & _val_col_w & "px;color:#F87171;font-size:14px;font-weight:700;text-align:left;'>" & _fmtA & "</div>" &
            "</div>",
        "",
        [tx_comp], DESC
    )
RETURN
    "<div style='font-family:Segoe UI,Arial,sans-serif;background:transparent;padding:16px 8px;'>" &
        _header & _rows &
    "</div>"
```

---
## 24. Checklist de validation

### Import & modèle
- [ ] 5 fichiers CSV importés (apprenants, parcours, inscriptions_analytics, paiements, apprenants_risque_scores)
- [ ] Auto Date/Time désactivé
- [ ] Table `Calendrier` créée et marquée comme table de dates
- [ ] Table `_Mesures` créée (colonne Value masquée)
- [ ] Table `Pays Affichage` créée (DATATABLE)
- [ ] 6 relations actives (Single direction partout)
- [ ] Visuel HTML Content installé depuis AppSource

### Colonnes calculées (4)
- [ ] `Calendrier[Jour Semaine Nom]` (trié par `Jour Semaine Ordre`)
- [ ] `Calendrier[Jour Semaine Ordre]`
- [ ] `apprenants[Tranche Age]` (trié par `Ordre Tranche Age`)
- [ ] `apprenants[Ordre Tranche Age]`
- [ ] `apprenants_risque_scores[Apprenant Nom Complet]`
- [ ] `apprenants_risque_scores[Parcours Court]`

### Mesures DAX par dossier (~45 mesures au total)

**📂 Pédagogie (8)** : `Nb Inscriptions`, `Taux Completion`, `Taux Abandon`, `CSAT Moyen`, `Nb Inscriptions Instructeur`, `Quadrant Instructeur`, `Pire Parcours du Domaine`, `Pire Parcours Taux`, `Pire Parcours Label`

**📂 Démographie (6)** : `% Apprenants Premium`, `Inscriptions par Apprenant`, `% Distribution Ages`, `Couleur Tranche Age`, `% Apprenants par Pays`, `Couleur Pays Affichage`

**📂 Acquisition (2)** : `% Apprenants par Canal`, `Couleur Canal Acquisition`

**📂 Paiement (2)** : `% Paiements par Methode`, `Couleur Methode Paiement`

**📂 Business (2)** : `Revenu Genere`, `Top 5 Parcours Rang`

**📂 Top 5 Parcours (15)** : `Top1 Nom`, `Top1 Revenu`, `Top1 Inscrits`, ..., `Top5 Nom`, `Top5 Revenu`, `Top5 Inscrits`

**📂 Alerte Décrochage (13)** : `Nb Apprenants Alerte`, `Nb Apprenants Risque Total`, `Phrase Alerte Decrochage`, `Taux Alerte Decrochage Label`, `Nb Apprenants Score > 0.80`, `Nb Apprenants Score 0.60 - 0.80`, `Nb Apprenants Score 0.30 - 0.60`, `Label Apprenants Score > 0.80`, `Label Apprenants Score 0.60 - 0.80`, `Label Apprenants Score 0.30 - 0.60`, `Action Recommandee`, `Couleur Action`, `Couleur Score Risque`, `Inactivite Label`, `Progression Label`

**📂 HTML Content (1)** : `Top 5 Parcours Completion vs Abandon HTML`

### Valeurs attendues sans filtre

| Mesure | Valeur |
|---|---|
| `[Nb Inscriptions]` (page couverture) | 6 456 |
| `[Taux Completion]` (page couverture) | 33,3% |
| `[Revenu Genere]` (page couverture) | 749M FCFA |
| `DISTINCTCOUNT(apprenants[apprenant_id])` | 4 500 |
| Page 1 `[Nb Inscriptions]` (filtré 2024) | 3 846 |
| Page 1 `[Taux Completion]` (filtré 2024) | 37,0% |
| Page 1 `[Taux Abandon]` (filtré 2024) | 19,9% |
| Page 1 `[CSAT Moyen]` (filtré 2024) | 4,23 |
| Page 2 `[% Apprenants Premium]` | 38% |
| Page 2 `[Inscriptions par Apprenant]` | 1,4 |
| Page 5 `[Phrase Alerte Decrochage]` | 1 613 apprenants en alerte de decrochage sur 2 095 en cours |
| Page 5 `[Taux Alerte Decrochage Label]` | 77,0% taux d'alerte |

### Pages
- [ ] **Page 0 Couverture** : brand hero + 4 KPI snapshots + menu navigation horizontal
- [ ] **Page 1 Vue executive** : 4 KPI + combo Inscriptions/Abandon + Donut domaines + Top 5 HTML
- [ ] **Page 2 Apprenants** : 3 KPI + Bar pays + Histogramme âges + Bar canal + Bar méthode paiement
- [ ] **Page 3 Parcours** : Scatter performance instructeurs + Heatmap activité + 4 cards Taux abandon
- [ ] **Page 4 Revenus** : 3 KPI + Stacked bar par domaine + Bar méthodes paiement + 5 cards Top parcours
- [ ] **Page 5 Alertes ML** : Bandeau alerte + 3 cards actions + Tableau Top apprenants

### Navigation
- [ ] Sidebar DataProjectLab Academy sur les 5 pages dashboard
- [ ] Bouton retour à l'accueil (icône flèche) en bas de chaque sidebar
- [ ] Slicers Période + Domaine synchronisés sur les 5 pages dashboard
- [ ] Page Couverture sans sidebar ni slicers

---
## 25. Storytelling — Présentation 5 minutes au comité de direction

### Structure narrative Problème → Cause → Solution → Impact

#### Minute 1 — Vue d'ensemble (Page 1)

> *"DataProjectLab Academy a généré **3 846 inscriptions sur 2024** avec un taux de complétion de **37%** (objectif fixé à 40%, écart -3pts). Le CSAT moyen est de **4,23/5** sur les parcours terminés — solide. Mais le **taux d'abandon de 19,9%** reste structurel, avec une saisonnalité marquée : pic à 22,1% en juillet, creux à 17,6% en décembre. Notre Top 5 des parcours par complétion est dominé par **HTML CSS JavaScript (40,7%)** et **SQL & Bases de Donnees (39,6%)** — les fondamentaux techniques."*

#### Minute 2 — Profil de la base apprenants (Page 2)

> *"Notre base 2024 compte **2 144 apprenants actifs** avec en moyenne **1,4 parcours/apprenant** (faible cross-sell) et **38% en Premium**. Géographiquement, on est dominants en **Côte d'Ivoire (40%)** et **Sénégal (20%)** — les autres pays restent à conquérir. Côté tranche d'âge : **60% ont entre 23 et 32 ans** — coeur de cible. Les canaux d'acquisition montrent un mix sain : **Organique 35%** (preuve d'attractivité de marque), **Réseaux sociaux 29%**. Côté paiement : **50% en Mobile Money** — il faut absolument optimiser ce flow."*

#### Minute 3 — Performance pédagogique (Page 3)

> *"La matrice instructeurs identifie **3 Top Performers** : Ndoye Cheick, Diallo Fatoumata et Traore Eric — à mettre en avant. **2 instructeurs Coach CSAT** ont d'excellents avis mais peinent sur la complétion (Bamba Clarisse, Konaté Aida) — leur transmission de savoir est forte mais ils manquent de mécaniques d'engagement. Sur les 4 domaines, le pire taux d'abandon est **Marketing Digital (21,2% moyen, pic à 21,9% sur le parcours Marketing Digital)** — domaine prioritaire à retravailler."*

#### Minute 4 — Performance financière (Page 4)

> *"Sur la période, on cumule **749 M FCFA (≈ 1,15 M EUR)** de revenu, panier moyen **89 554 FCFA**. La répartition mensuelle est stable, sans saisonnalité financière marquée. **Top 5 parcours par revenu** : React & Node.js (120M), Machine Learning (109M), Python Web Django (94M), Data Analyse Python (72M), Power BI (64M) — **les parcours Tech génèrent l'essentiel du CA** avec un alignement clair entre prix premium et popularité."*

#### Minute 5 — L'enjeu critique : décrochage (Page 5)

> *"Le modèle ML a identifié **1 613 apprenants en alerte de décrochage sur 2 095 en cours — 77% de taux d'alerte**. Le plan d'action est triple :
> 1. **~150 apprenants score >0.80** → appel téléphonique sous 48h (intervention humaine)
> 2. **~150 apprenants score 0.60-0.80** → email personnalisé sous 7j (relance ciblée)
> 3. **~1 300 apprenants score 0.30-0.60** → notification push automatisée (rappel doux)
> Le tableau Top apprenants à risque liste les cas critiques nominativement — **Adjoua Coulibaly, Serge Diop, Yao Mbaye** sont à appeler en priorité."*

### Impact estimé du plan d'action sur 90 jours

- **Réduction de 30% du taux d'abandon** sur les 1 613 apprenants alertés (intervention ML préventive)
- **+5 pts de complétion globale** (de 37% à 42% — au-dessus de l'objectif 40%)
- **+ 100 M FCFA de revenu protégé** (apprenants qui termineront et achèteront le parcours suivant)
- **Objectif Q+1** : ramener le taux d'alerte de 77% à 60%

---

> L'apprenant doit pouvoir répondre à la question : **"Quelle est la priorité opérationnelle pour DataProjectLab Academy ce trimestre ?"**
> La réponse est contenue dans les 6 pages du dashboard — pas ailleurs.

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.